# Bonus B4: How to build a crew with CrewAI

**Notebook 2 of 2: CrewAI at the Chinook desk**

## Goal

Take the newsroom's two-person pattern to the Chinook music shop's support desk. A support rep passes on a request:
*Puja Srivastava says invoice 229 was charged to her card twice and asks for her money back.* A billing clerk with two
database lookups finds the customer and the invoice; a refund adviser with no tools advises the rep, as a typed
object our code can read. Alone, the adviser has no amount to refund. Given the clerk's lookup through `context`, it
advises a refund of the invoice's total, and that advice breaks the shop's rule that the desk refunds at most $10
itself. A guardrail in our code refuses it, and the adviser corrects it to a review by a person.

## What you will learn

- How to give each specialist only the tools its task needs, and pass the facts to one that has no tools through
  `context`.
- Why every lookup opens its own database connection: CrewAI runs the tool calls of one reply at the same time, in
  threads.
- How to ask a task for a typed result with `output_pydantic`, and read its fields.
- How a guardrail enforces a rule a type cannot: a valid `RefundAdvice` can still break the refund policy.
- How to count a crew's requests and tokens and price them from `models.env`.

## Setup

The first cell is the same in every lab: it finds the repository folder, reads the model name from `models.env`, and
takes the key from `.env` without printing it (lab 00 explains it line by line). Select this folder's `.venv` as the
kernel: CrewAI has its own environment in this lab.

In [1]:
from pathlib import Path
import os
from dotenv import dotenv_values

# The repository folder is the first folder, from here upwards, that holds models.env
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "models.env").exists())

# Models and prices come from models.env; a non-empty value in .env replaces one of them
settings = dotenv_values(ROOT / "models.env")
local = dotenv_values(ROOT / ".env")
settings.update({name: value for name, value in local.items() if name in settings and value})

# The key comes from the environment or .env, and is never printed
os.environ.setdefault("OPENAI_API_KEY", local.get("OPENAI_API_KEY") or "")
assert os.environ["OPENAI_API_KEY"], "Put OPENAI_API_KEY in .env at the repository folder (copy .env.example)."

MODEL = settings["MODEL_DEFAULT"]
EFFORT = settings["REASONING_EFFORT"] or None
print("model:", MODEL, "| reasoning effort:", EFFORT, "| key found: yes")

model: gpt-6-luna | reasoning effort: none | key found: yes


The CrewAI switches go before the import, as in notebook 1: no telemetry, no hosted tracing, no version check, a log
without colours, and CrewAI's own files in this lab's `outputs/` folder.

In [2]:
# CrewAI reads these when it is imported, so they come first
os.environ["CREWAI_DISABLE_TELEMETRY"] = "true"      # no usage statistics sent to CrewAI
os.environ["CREWAI_TRACING_ENABLED"] = "false"       # no traces sent to CrewAI's hosted platform
os.environ["CREWAI_DISABLE_VERSION_CHECK"] = "true"  # no check for a newer CrewAI on every start
os.environ["NO_COLOR"] = "1"                         # the crew's log prints without colours

# CrewAI's own files (the last run's task outputs) go to this lab's outputs/ folder, which git ignores
(Path.cwd() / "outputs").mkdir(exist_ok=True)
os.environ["CREWAI_STORAGE_DIR"] = str(Path.cwd() / "outputs")

Every agent gets its own `LLM`, with the model and the reasoning effort from `models.env` and a 60-second limit on
each request. An `LLM` object counts the requests made through it, so two agents sharing one would make the crew's
count too high.

In [3]:
from crewai import LLM, Agent, Task, Crew


def new_llm():
    return LLM(model=f"openai/{MODEL}", api="responses", reasoning_effort=EFFORT, timeout=60)

---

## Part A: Build the clerk's two lookups and call them by hand

The clerk needs two facts from the shop's database, `data/chinook.db`: who the customer is, and what the invoice says.
Each lookup is a CrewAI tool, made with `@tool` from an ordinary function, as in notebook 1.

Each one opens **its own** read-only connection to the database and closes it when it returns:

- `f"file:{DB}?mode=ro"` with `uri=True` opens the file read-only, so no lookup can change it.
- `with closing(...) as con:` closes the connection at the end of the block, even after an error.

Why not one connection for the whole notebook? When a model asks for two tools in one reply, CrewAI runs them at the
same time, each in its own thread. By default `sqlite3` refuses outright to use a
connection in a thread other than the one that opened it. The usual way to let other threads use it is to open the
connection once with `check_same_thread=False`; one connection shared between threads then returns empty results,
another customer's row, or errors. Part C shows the two threads. This notebook does not run a shared connection: Lab
18's extra notebook ran 1,000 lookups, five at a time, on one shared connection, and 112 went wrong. A connection per lookup has none of those problems, and opening a local SQLite file is fast.

Each tool prints a `TOOL CALLED` line with the name of the thread it runs in, and a lookup that finds nothing says so
in words: after an empty result, the model tends to fill the gap with a guess.

In [4]:
import json
import sqlite3
import threading
from contextlib import closing
from crewai.tools import tool

DB = ROOT / "data" / "chinook.db"


@tool("find_customer")
def find_customer(name: str) -> str:
    """Find a shop customer by full name. Returns the customer_id, name and country."""
    print(f"🔎 TOOL CALLED: find_customer({name}) in {threading.current_thread().name}")
    with closing(sqlite3.connect(f"file:{DB}?mode=ro", uri=True)) as con:
        rows = con.execute(
            "SELECT CustomerId, FirstName || ' ' || LastName, Country FROM customers "
            "WHERE FirstName || ' ' || LastName = ?", (name,)).fetchall()
    if not rows:
        return f"no match for customer {name}"
    return json.dumps([{"customer_id": r[0], "name": r[1], "country": r[2]} for r in rows], ensure_ascii=False)


@tool("get_invoice")
def get_invoice(invoice_id: int) -> str:
    """Look up one invoice by its number. Returns the customer_id, the date and the total in US dollars."""
    print(f"🔎 TOOL CALLED: get_invoice({invoice_id}) in {threading.current_thread().name}")
    with closing(sqlite3.connect(f"file:{DB}?mode=ro", uri=True)) as con:
        rows = con.execute(
            "SELECT InvoiceId, CustomerId, date(InvoiceDate), Total FROM invoices WHERE InvoiceId = ?",
            (invoice_id,)).fetchall()
    if not rows:
        return f"no match for invoice {invoice_id}"
    invoice_id, customer_id, day, total = rows[0]
    return json.dumps({"invoice_id": invoice_id, "customer_id": customer_id, "date": day, "total": total})

We call each lookup ourselves first, including one invoice that does not exist.

In [5]:
print(find_customer.run(name="Puja Srivastava"))
print(get_invoice.run(invoice_id=229))
print(get_invoice.run(invoice_id=9999))

🔎 TOOL CALLED: find_customer(Puja Srivastava) in MainThread
[{"customer_id": 59, "name": "Puja Srivastava", "country": "India"}]
🔎 TOOL CALLED: get_invoice(229) in MainThread
{"invoice_id": 229, "customer_id": 59, "date": "2011-09-30", "total": 13.86}
🔎 TOOL CALLED: get_invoice(9999) in MainThread
no match for invoice 9999


Puja Srivastava is customer 59, in India. Invoice 229 belongs to customer 59, is dated 2011-09-30 and comes to
$13.86. Invoice 9999 does not exist, and the tool says so in a sentence. The `MainThread` at the end of each
`TOOL CALLED` line is the notebook's own thread: we called the lookups ourselves, one after the other.

**Checkpoint A.** Puja Srivastava is customer 59, in India; invoice 229 is hers (customer 59), dated 2011-09-30, for
$13.86; invoice 9999 gets a "no match" sentence. Each line ran in `MainThread`.

---

## Part B: Ask the refund adviser with no facts

The refund adviser has no tools. Its backstory carries the desk's habit for this kind of request: *Duplicate charges
are refunded in full.*

### A typed answer: the example first

The rep's screen needs the advice as fields, not as a paragraph: whose refund, which invoice, how much, what to do,
and why. One piece of advice, written by hand, looks like this:

In [6]:
example = {
    "customer_name": "Puja Srivastava",
    "invoice_id": 229,
    "amount": 13.86,
    "action": "refund",
    "reason": "Invoice 229 was charged twice; duplicate charges are refunded in full.",
}
print(example["action"], example["amount"])

refund 13.86


`RefundAdvice` describes that shape as a Pydantic model: each field and its type. `action` is a `Literal`, so it can
only be `"refund"` or `"human_review"`, the two things the desk can do. `RefundAdvice(**example)` checks our example
against it.

`output_pydantic=RefundAdvice` on a task asks CrewAI to return the task's result as a `RefundAdvice` object. After
the run, `result.pydantic` is the last task's `RefundAdvice`, and each `TaskOutput` in `result.tasks_output` has its own `.pydantic`.

In [7]:
from typing import Literal
from pydantic import BaseModel


class RefundAdvice(BaseModel):
    customer_name: str
    invoice_id: int
    amount: float
    action: Literal["refund", "human_review"]
    reason: str


print(RefundAdvice(**example))

customer_name='Puja Srivastava' invoice_id=229 amount=13.86 action='refund' reason='Invoice 229 was charged twice; duplicate charges are refunded in full.'


Now the adviser, its task and a crew of one. `{request}` in the description is filled from `inputs`.

In [8]:
REQUEST = "Puja Srivastava says invoice 229 was charged to her card twice and asks for her money back."
ADVISE = "A support rep passes on this request: {request}\nAdvise the rep what to do."


def new_adviser():
    return Agent(
        role="Refund adviser",
        goal="Advise the support rep on what to do with a refund request.",
        backstory="You work at the support desk of the Chinook music shop. Duplicate charges are refunded in full.",
        llm=new_llm(),
    )


adviser = new_adviser()
advice_task = Task(description=ADVISE, expected_output="The advice.", agent=adviser, output_pydantic=RefundAdvice)

result = await Crew(agents=[adviser], tasks=[advice_task]).akickoff(inputs={"request": REQUEST})

advice = result.pydantic
print(type(advice).__name__)
for field, value in advice.model_dump().items():
    print(f"  {field}: {value}")
print("model requests:", result.token_usage.successful_requests)

RefundAdvice
  customer_name: Puja Srivastava
  invoice_id: 229
  amount: 0.0
  action: human_review
  reason: The duplicate charge is alleged, but the invoice amount and duplicate charge have not been verified. Review the billing records and refund the duplicate charge in full if confirmed.
model requests: 1


The answer is a `RefundAdvice` with every field filled, and the fields show what the adviser did not have. It took the
name and the invoice number from the request, but the amount is `0.0` and the action is `human_review`: the reason says
the amount and the duplicate charge have not been verified. No figure from the database is in it, because nothing
from the database reached this agent. **1 model request**: no tools, one reply.

**Checkpoint B.** The example passes `RefundAdvice`. The adviser alone returns a `RefundAdvice` with no real amount
and no facts from the database.

---

## Part C: Pass the clerk's lookup to the adviser with `context`

The billing clerk gets the two lookups and a task that asks for the facts. The adviser keeps no tools and gets the
clerk's task as its `context`, the same pattern as the newsroom.

![One run of a two-task crew. Our code starts the crew. In task 1 the model asks for the tool, our tool function runs and returns the result, and the model writes it up. Task 1's output is passed on as context to task 2, where the model writes from the context with no tools. Our guardrail function then accepts the output, which becomes the CrewOutput, or refuses it and sends the reason back to task 2's model.](resources/b4_crew_tasks_and_context.png)

Dark boxes are the model's work, light green is our own code, white is text passed along. Here task 1 is the clerk's
lookup, task 2 the adviser's advice; the guardrail is Part D.

Watch the thread names in the `TOOL CALLED` lines.

In [9]:
def new_clerk():
    return Agent(
        role="Billing clerk",
        goal="Look up customers and invoices in the shop's database and report exactly what it holds.",
        backstory="You work at the support desk of the Chinook music shop.",
        tools=[find_customer, get_invoice],
        llm=new_llm(),
    )


LOOKUP = "A support rep passes on this request: {request}\nLook up the customer and the invoice."
FACTS = "The customer's id and name, and the invoice's number, date and total."

clerk, adviser = new_clerk(), new_adviser()
lookup_task = Task(description=LOOKUP, expected_output=FACTS, agent=clerk)
advice_task = Task(description=ADVISE, expected_output="The advice.", agent=adviser,
                   context=[lookup_task], output_pydantic=RefundAdvice)

result = await Crew(agents=[clerk, adviser], tasks=[lookup_task, advice_task]).akickoff(inputs={"request": REQUEST})

🔎 TOOL CALLED: find_customer(Puja Srivastava) in ThreadPoolExecutor-7_0🔎 TOOL CALLED: get_invoice(229) in ThreadPoolExecutor-7_1



Look at the two `TOOL CALLED` lines. The clerk asked for both lookups in one reply, and they ran in two different
threads, `ThreadPoolExecutor-7_0` and `ThreadPoolExecutor-7_1`, at the same time: in our run their prints even landed
on the same line, one starting before the other had finished printing. This is why each lookup opens its own
connection.

What did each specialist hold? The tools come from each agent's `tools` list; the context from each task's
`prompt_context`.

In [10]:
for agent in [clerk, adviser]:
    print(f"{agent.role}: tools {[t.name for t in agent.tools]}")
print()
print("clerk's lookup:", result.tasks_output[0].raw)
print("context given to the adviser:", advice_task.prompt_context)
print()
advice = result.pydantic
for field, value in advice.model_dump().items():
    print(f"  {field}: {value}")
print("model requests:", result.token_usage.successful_requests)

Billing clerk: tools ['find_customer', 'get_invoice']
Refund adviser: tools []

clerk's lookup: Customer ID: 59  
Customer name: Puja Srivastava  
Invoice number: 229  
Invoice date: 2011-09-30  
Invoice total: $13.86
context given to the adviser: Customer ID: 59  
Customer name: Puja Srivastava  
Invoice number: 229  
Invoice date: 2011-09-30  
Invoice total: $13.86

  customer_name: Puja Srivastava
  invoice_id: 229
  amount: 13.86
  action: refund
  reason: Duplicate charges are refunded in full; invoice 229 total is $13.86.
model requests: 3


The clerk holds both lookups and the adviser none. The clerk's lookup is five lines of facts, and the adviser's
`prompt_context` is those same five lines: the adviser's request carries nothing else about Puja's invoice. The advice now
names invoice 229 and the amount **13.86**, the invoice's total from the database, with the action `refund`.
**3 model requests for two tasks**: two for the clerk (both tool calls in one reply, then the facts) and one for the
adviser.

**Checkpoint C.** Two `TOOL CALLED` lines in two different `ThreadPoolExecutor` threads; the clerk holds both
lookups and the adviser none; the adviser's context is the clerk's lookup; the advice names Puja Srivastava and
invoice 229, with the amount 13.86 and the action `refund`; 3 model requests for two tasks.

---

## Part D: Refuse advice that breaks the refund policy

The shop's rule: **the desk refunds at most $10 by itself; a larger refund goes to a person** (`human_review`). The
advice from Part C is a valid `RefundAdvice` (every field is there, with the right type) and it breaks that rule.
A type checks the shape of an answer; whether the answer is allowed is a separate question, and the rule is exact, so
it belongs in our code.

`check_advice` reads `output.pydantic`, the `RefundAdvice` object, prints what it decided and returns the pair CrewAI
expects: `(True, value)` to accept, `(False, reason)` to refuse and send the reason back to the adviser.
`REFUND_LIMIT` is ours: the adviser's prompt never mentions it. On accept we return the text, `output.raw`; CrewAI
builds the `RefundAdvice` again from that text, as the fields printed after the crew's run below show.

In [11]:
REFUND_LIMIT = 10.00


def check_advice(output):
    advice = output.pydantic
    if advice.action == "refund" and advice.amount > REFUND_LIMIT:
        print(f"❌ refused: refund of ${advice.amount:.2f}")
        return (False, f"A refund above ${REFUND_LIMIT:.2f} needs a person: the action must be human_review.")
    print(f"✅ accepted: {advice.action}, ${advice.amount:.2f}")
    return (True, output.raw)

First on Part C's advice, by hand, before any task uses it:

In [12]:
print(check_advice(advice_task.output))

❌ refused: refund of $13.86
(False, 'A refund above $10.00 needs a person: the action must be human_review.')


The advice from Part C, a refund of $13.86, is refused, with the reason the adviser would receive.

Now on the task, with `guardrail=check_advice`, and fresh agents for a new run.

In [13]:
clerk, adviser = new_clerk(), new_adviser()
lookup_task = Task(description=LOOKUP, expected_output=FACTS, agent=clerk)
advice_task = Task(description=ADVISE, expected_output="The advice.", agent=adviser,
                   context=[lookup_task], output_pydantic=RefundAdvice, guardrail=check_advice)

result = await Crew(agents=[clerk, adviser], tasks=[lookup_task, advice_task]).akickoff(inputs={"request": REQUEST})

print()
for field, value in result.pydantic.model_dump().items():
    print(f"  {field}: {value}")
print("model requests:", result.token_usage.successful_requests)

🔎 TOOL CALLED: find_customer(Puja Srivastava) in ThreadPoolExecutor-15_0
🔎 TOOL CALLED: get_invoice(229) in ThreadPoolExecutor-15_1


❌ refused: refund of $13.86


✅ accepted: human_review, $13.86

  customer_name: Puja Srivastava
  invoice_id: 229
  amount: 13.86
  action: human_review
  reason: Invoice 229 was charged twice. Duplicate charges are refunded in full, but a refund above $10.00 needs a person to review.
model requests: 4


The lookups again ran in two threads. The adviser's first advice was a refund of $13.86 and was refused; the reason
went back to the adviser, and its second advice, `human_review` for the same $13.86, was accepted. The final reason
repeats the rule it was sent. The retry cost **one more model request**: 4, against 3 in Part C.

What the ✅ establishes: the action and the amount together keep to the $10 rule. It does not check that the amount is
the invoice's total, or that the invoice is Puja's; the lookups and the context are what make those right.

### What the run cost

`token_usage` adds up the whole crew: both agents, every attempt. We price it with the input and output prices from
`models.env` (per million tokens).

In [14]:
usage = result.token_usage
price_in = float(settings["PRICE_INPUT_PER_MILLION"])
price_out = float(settings["PRICE_OUTPUT_PER_MILLION"])
cost = (usage.prompt_tokens * price_in + usage.completion_tokens * price_out) / 1_000_000

print("model requests:", usage.successful_requests)
print("input tokens:  ", usage.prompt_tokens, "| of which cached:", usage.cached_prompt_tokens)
print("output tokens: ", usage.completion_tokens)
print(f"cost: ${cost:.5f}")

model requests: 4
input tokens:   1692 | of which cached: 0
output tokens:  225
cost: $0.00028


**4 requests, 1,692 input tokens and 225 output tokens**, about three hundredths of a US cent at the `models.env`
prices. None of the input was cached: OpenAI caches a prompt only from 1,024 tokens, and every request here is shorter.
Some of these input tokens come from splitting the work into two tasks: the five lines of facts are sent twice, once
as the clerk's tool results and again as the adviser's context.

**Checkpoint D.** `check_advice` refuses Part C's refund of $13.86 by hand. On the task, a ❌ line for the refund, then
a ✅ line for `human_review` with the same amount; the final advice is `human_review`; 4 model requests, and a cost
well under one US cent.

---

## What you can do now

- Give each specialist only the tools its task needs, and pass facts to one with no tools through `context`
  (Part C).
- Open a database connection per lookup, and say why CrewAI's threads make that necessary (Parts A and C).
- Ask a task for a typed result with `output_pydantic`, starting from an example of it (Part B).
- Enforce a rule a type cannot, with a guardrail that refuses the output and sends the reason back (Part D).
- Count a crew's requests and tokens and price them from `models.env` (Part D).

## Troubleshooting

| What you see | Why | Fix |
|---|---|---|
| `RuntimeError: Agent execution was invoked synchronously from within a running event loop.` | `crew.kickoff()` in a notebook | write `await crew.akickoff(...)` |
| the advice has an invented customer id or amount | a lookup returned nothing, or the task had no `context` | call the lookups by hand (Part A) and check the adviser's `prompt_context` |
| `StopIteration` in the setup cell | the notebook was started outside the repository folder, so no `models.env` was found | open the notebook from `bonus/B4_CrewAI_Crew` and run the setup cell again |
| `sqlite3.OperationalError: unable to open database file` | `data/chinook.db` is missing from the repository folder | restore `data/chinook.db` (it comes with the repository) and run the cell again |
| `Exception: Task failed guardrail validation after 3 retries.` | every attempt broke the rule | read the ❌ lines; add the rule to the task's description as well, so the first attempt has it |

## Challenge

The policy lets small refunds through. Puja's invoice 284 is for $8.91.

1. Send the crew of Part D the request *Puja Srivastava says invoice 284 was charged to her card twice and asks for
   her money back.*
2. Print the advice's fields, the ✅ or ❌ lines and the model requests.

Expected: one ✅ line for a refund of $8.91 at the first attempt, and one model request fewer than Part D's run. Then
look at what the adviser wrote as the `reason`: does it say why no person is needed?

The worked solution, run with its outputs, is in `solved/b4b_crewai_at_the_desk_solution.ipynb`.

## Resources

- [Tasks](https://docs.crewai.com/en/concepts/tasks): `context`, `output_pydantic`, guardrails and their retries
- [Agents](https://docs.crewai.com/en/concepts/agents): role, goal, backstory, tools
- [Tools](https://docs.crewai.com/en/concepts/tools): the `@tool` decorator
- [Crews](https://docs.crewai.com/en/concepts/crews): `akickoff` and the crew's usage
- [Pydantic models](https://docs.pydantic.dev/latest/concepts/models/): `BaseModel`, fields and validation
- [Python `sqlite3`](https://docs.python.org/3/library/sqlite3.html): connections, and the read-only `mode=ro` URI

Back to notebook 1, `b4a_meet_crewai.ipynb`, or to [the lab's README](README.md).